# خطوة الذكاء الاصطناعي: تصنيف مشاعر المراجعات

هذا الجزء الوحيد اللي سويته برا Excel. دربت نموذج بسيط يقرأ نص المراجعة العربية ويعطي **احتمال إنها إيجابية** من 0 إلى 1. بعدها صدّرت النتيجة لـ Excel، وكل التحليل واللوحة هناك.

**البيانات:** HARD (Hotel Arabic-Reviews Dataset): مراجعات عربية من Booking.com، أغلبها لفنادق مكة والمدينة.

In [1]:
!git clone -q --depth 1 https://github.com/elnagara/HARD-Arabic-Dataset
!unzip -o -q HARD-Arabic-Dataset/data/balanced-reviews.zip

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

d = pd.read_csv('balanced-reviews.txt', sep='\t', encoding='utf-16')
d['review'] = d['review'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
d['label'] = (d['rating'] >= 4).astype(int)   # 4-5 stars = positive, 1-2 = negative
print(d.shape)
d['rating'].value_counts()

(105698, 8)


rating
2    38467
4    26450
5    26399
1    14382
Name: count, dtype: int64

### العينة اللي بحللها في Excel
أخذت أكبر 15 فندق، و140 مراجعة من كل فندق، يعني 2,100 مراجعة. وهذي ما تدخل في التدريب أبداً، عشان أقيس النموذج على مراجعات ما شافها.

In [3]:
top = d['Hotel name'].value_counts().head(15).index
sample = d[d['Hotel name'].isin(top)].groupby('Hotel name').sample(140, random_state=1)
train = d.drop(sample.index)
print(len(sample), len(train))

2100 103598


### النموذج
استخدمت **TF-IDF على مستوى الحروف** مع **Logistic Regression**. اخترت الحروف بدل الكلمات لأن المراجعات فيها لهجات وأخطاء إملائية كثيرة، مثل «نظافه» و«نظافة»، والحروف تتعامل معها أحسن.

In [4]:
tr, va = train_test_split(train, test_size=0.1, random_state=0, stratify=train['label'])
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=300000, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, C=4).fit(vec.fit_transform(tr['review']), tr['label'])

pred = clf.predict(vec.transform(va['review']))
print(f"validation accuracy: {accuracy_score(va['label'], pred):.3f} | F1: {f1_score(va['label'], pred):.3f}")

validation accuracy: 0.947 | F1: 0.947


In [5]:
sample['p_pos'] = clf.predict_proba(vec.transform(sample['review']))[:, 1].round(3)
print(f"accuracy on the excel sample: {accuracy_score(sample['label'], (sample['p_pos'] >= .5).astype(int)):.3f}")
sample[['Hotel name', 'user type', 'nights', 'rating', 'review', 'p_pos']].to_csv('reviews_for_excel.csv', index=False, encoding='utf-8-sig')

accuracy on the excel sample: 0.937


**النتيجة:** دقة حوالي 94% على مراجعات ما شافها النموذج. والملف `reviews_for_excel.csv` هو اللي دخل Excel.